In [ ]:
import h5py
import os
import numpy as np
import matplotlib.pyplot as plt
from scipy.io import loadmat, savemat
from tqdm import tqdm

basepath = "/mnt/DATA/NonLinearMI/EEG_el_so_BLP/"
sub=lambda x: "Sub_{}".format(x)
filename = ["EEG_bands.mat", "electrode_BLP.mat", "source_BLP.mat"]

In [ ]:
# '#refs#', 'EEG_bands'
which_file = 0
refs=[]
infs=[]
inname=os.path.splitext(filename[which_file])[0]
for pat in range(1,51):
    print("*** pat {} ***".format(pat))
    mat = h5py.File(os.path.join(basepath, sub(pat), filename[which_file]))
    print("refs", np.array(mat['#refs#']).shape)
    print({k:mat['#refs#'][k].shape for k in mat['#refs#'].keys()})
    refs.append(np.array(mat['#refs#']).shape[0])
    print(inname, mat[inname].shape)
    infs.append(mat[mat[inname][0,0]].shape[0])
    for i in range(9):
        print(i, mat[mat[inname][0,i]].shape)
        inf = mat[mat[inname][0,i]].shape[0]
        for j in range(inf):
            print("\t",j,mat[mat[mat[inname][0,i]][j,0]].shape)

In [ ]:
which_file = 0
totlen = []
inname=os.path.splitext(filename[which_file])[0]
for pat in tqdm(range(1,51), desc="Subj"):
    mat = h5py.File(os.path.join(basepath, sub(pat), filename[which_file]))
    n_pieces = mat[mat[inname][0,0]].shape[0]
    band=0
    tmp = []
    for piece in range(n_pieces):
        tmp.append(mat[mat[mat[inname][0,band]][piece,0]].shape[0])
    totlen.append(sum(tmp))    
print(min(totlen))

In [ ]:
plt.hist(totlen, bins="auto")
plt.title(min(totlen))
plt.show()

In [ ]:
which_file = 2
maxlendipi = []
totlen = []
inname=os.path.splitext(filename[which_file])[0]
new_datasets_raw=[[] for __ in range(9)]
for pat in tqdm(range(1,51), desc="Subj"):
    mat = h5py.File(os.path.join(basepath, sub(pat), filename[which_file]))
    n_pieces = mat[mat[inname][0,0]].shape[0]
    for band in range(9):
        tmp = []
        for piece in range(n_pieces):
            tmp.append(mat[mat[mat[inname][0,band]][piece,0]])
        tmp_arr = np.concatenate(tmp, axis=0)
        new_datasets_raw[band].append(tmp_arr[:300])

for band, dataset in tqdm(enumerate(new_datasets_raw), desc="Band"):
    new_dataset = np.stack(dataset, -1)
    outfile = inname+"_band{}.mat".format(band+1)
    savemat(os.path.join(basepath, outfile), {inname:new_dataset})

In [ ]:
which_file = 0
inname=os.path.splitext(filename[which_file])[0]
new_datasets_raw=[[] for __ in range(9)]
mat = h5py.File(os.path.join(basepath, sub(1), filename[which_file]))
for i in range(9):
    plt.plot(mat[mat[mat[inname][0,i]][0,0]][:500,0]+i, label=i)
    # plt.plot(np.fft.fft(mat[mat[mat[inname][0,i]][0,0]][:,0])[:800], label=i)

plt.legend()
plt.show()